# 1. ICA per participant

Follows `ana01_ICA.m` of the reference pipeline (Stropahl et al., 2018; `notes/papers/eeg_data_sheet.PDF`), adapted to the RiseTogether BIDS dataset (read with `rt_bids`).

**Run once per participant.** Then run `02_corrmap.ipynb` (all participants together), then `03_clean.ipynb` (per participant).

**Flow:** 1. load (mastoids removed) → 2. events → 3. mark breaks + crop → 4. bad channels (interpolated), then re-reference (`cfg.REFERENCE`) → 5. save the prepared data → 6. ICA training copy (low-pass 40 Hz → 250 Hz → high-pass 1 Hz) → 7. 1 s dummy epochs, reject unusual ones (joint probability > 3 SD) → 8. extended Infomax ICA → 9. save ICA and component plots → 10. a property plot (map, time course, spectrum) of every component

All settings are in `config.py`. Open questions about the adaptation: `ica_reference_questions.md`.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

import config as cfg  # settings; also puts rt_bids on the path
import rt_bids
from utils import find_bad_channels, firws_filter, jointprob_keep, plot_component_maps
%matplotlib inline

## 0. Participant

In [ ]:
import os

# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
OUT_DIR = cfg.out_dir(SUBJECT)
print(SUBJECT, f"({cfg.session_of(SUBJECT)})", "->", OUT_DIR)

## 1. Loading the raw data

One EEG recording per participant from the BIDS dataset (`sub-NNN/eeg/*.bdf`), read with `rt_bids.read_recording`. The channels already have electrode names and types (EEG, EOG2–4 as EOG), and channels marked `bad` in `channels.tsv` (at the ADC limit more than half of the time) are already in `raw.info["bads"]`. The mastoids (M1, M2) are removed.

In [ ]:
raw = rt_bids.read_recording(cfg.BIDS_ROOT, SUBJECT).load_data()
print(f"{SUBJECT}: {raw.info['nchan']} channels, {raw.times[-1]:.1f} s, {raw.info['sfreq']} Hz")
print("Marked bad in the dataset:", raw.info["bads"])

raw.drop_channels(cfg.DROP_CHANNELS)
raw.set_channel_types({ch: "eog" for ch in cfg.EOG_CHANNELS})
raw.set_montage(cfg.MONTAGE)

# lets rt_bids find this recording's session clock later (kept in the saved .fif)
raw.info["description"] = "rt_bids:" + str(cfg.bdf_path(SUBJECT).resolve())
print(f"EEG channels: {len(mne.pick_types(raw.info, eeg=True))}, EOG channels: {len(mne.pick_types(raw.info, eog=True))}")

## 2. Events

Every event of `events.tsv` is an annotation at its true sample in this recording (`rt_bids` leaves out the sync-only events). Skipped samples (`acq_gap`) are only counted (saved in `_ica_info.csv`).

In [ ]:
assert len(raw.annotations) > 0, f"{SUBJECT}: no annotations"

descriptions = pd.Series(raw.annotations.description)
n_acq_gaps = int(descriptions.str.startswith("acq_gap").sum())
print(f"{len(raw.annotations)} annotations; skipped-sample gaps (acq_gap): {n_acq_gaps}")

trial_starts = descriptions[descriptions.str.startswith(cfg.TRIAL_START)]
print(trial_starts.str.split("/").str[1].value_counts())

events, event_id = mne.events_from_annotations(raw, regexp="^TRIAL_(START|END)/")
fig = mne.viz.plot_events(events, event_id=event_id, sfreq=raw.info["sfreq"], first_samp=raw.first_samp)

### Check: are all files of the session complete?

Every participant should have the same number of trials per condition. Start times are not compared: the headsets were switched on one at a time.

In [ ]:
rows = []
for sub, r in rt_bids.read_session(cfg.BIDS_ROOT, cfg.session_of(SUBJECT)).items():
    starts = [d.split("/")[1] for d in r.annotations.description if d.startswith(cfg.TRIAL_START)]
    row = {"subject": sub, "sfreq": r.info["sfreq"], "n_channels": r.info["nchan"],
           "bads_in_dataset": " ".join(r.info["bads"])}
    row.update(pd.Series(starts, dtype=str).value_counts().to_dict())
    rows.append(row)
pd.DataFrame(rows)

## 3. Marking breaks and cropping to the task

The recording covers the whole session. Long gaps between trials (surveys, results screens, instructions, team reassignment) are marked as `BAD_break`, so ICA does not learn from them; then the data is cropped to the trials. The motor baseline is kept (`cfg.CONDITIONS`).

Breaks are marked **before** cropping, while the data still start at sample 0, so their onsets need no correction.

In [ ]:
def trial_times(raw, prefix):
    """Times (s, from the start of the data) of the TRIAL_START/ or TRIAL_END/ events of the kept conditions."""
    pattern = "^" + prefix + "(" + "|".join(cfg.CONDITIONS) + ")/"
    ev, _ = mne.events_from_annotations(raw, regexp=pattern, verbose="error")
    return np.sort((ev[:, 0] - raw.first_samp) / raw.info["sfreq"])


starts, ends = trial_times(raw, cfg.TRIAL_START), trial_times(raw, cfg.TRIAL_END)

break_onsets, break_durations = [], []
for end, next_start in zip(ends[:-1], starts[1:]):
    if next_start - end > cfg.BREAK_MIN_DURATION:
        break_onsets.append(end + 1.0)
        break_durations.append(next_start - end - 2.0)  # 1 s margin on each side

onset_offset = raw.first_time if raw.annotations.orig_time is not None else 0.0
breaks = mne.Annotations(onset=np.array(break_onsets) + onset_offset, duration=break_durations,
                         description="BAD_break", orig_time=raw.annotations.orig_time)
raw.set_annotations(raw.annotations + breaks)

raw.crop(tmin=max(starts.min() - 2.0, 0), tmax=min(ends.max() + 2.0, raw.times[-1]))
print(f"Breaks marked: {len(breaks)}, total {np.sum(break_durations):.1f} s of {raw.times[-1]:.1f} s")

## 3b. Marking bad stretches (added to the reference pipeline)

Some recordings have stretches where **all channels at once** jump to hundreds or thousands of µV (e.g. sub-013 in its last Team block: headset, cable or cap problems). ICA cannot remove these: they have no fixed scalp map, and their size would dominate the decomposition. So they are marked as `BAD_amplitude` here, before anything else uses the data: the units check, the bad-channel check and the ICA (the dummy epochs skip them, like the breaks). The marks are saved with the data, so `03_clean` and the later analysis see them too.

How: the EEG (1–40 Hz) is cut into 1 s windows; a window is bad when the **median** of its channel SDs is above `cfg.BAD_SEGMENT_SD_UV` (50 µV; clean data is about 10 µV). The median ignores a single noisy channel (that is step 4's job) and a blink on the frontal channels. The SDs are measured with the reference used later (common average, `cfg.REFERENCE`): with the mastoid reference, a noisy mastoid makes every channel look bad (sub-002), but the average reference removes that.

The table shows how much of each trial is lost; it is saved as `<sub>_bad_segments.csv`.

In [ ]:
# 1) median channel SD (1-40 Hz) in 1 s windows
win_s = cfg.BAD_SEGMENT_WINDOW
win = int(win_s * raw.info["sfreq"])                                         # samples per window
# measured with the reference used later (step 4): a noisy mastoid reference makes every channel look
# noisy, but the common average removes it (sub-002: 42% of trial time above 50 µV with mastoids, 0.4% with the average)
eeg = raw.copy().pick("eeg").filter(1, 40, verbose="error")
if cfg.REFERENCE == "average":
    eeg.set_eeg_reference("average", verbose="error")
eeg = eeg.get_data() * 1e6                                                   # µV, channels x samples
n_win = eeg.shape[1] // win
windows = eeg[:, :n_win * win].reshape(len(eeg), n_win, win)                 # channels x windows x samples
window_sd = np.median(windows.std(axis=2), axis=0)                           # one number per window
is_bad = window_sd > cfg.BAD_SEGMENT_SD_UV
del eeg, windows                                                             # free memory

# 2) mark the bad windows as BAD_amplitude (in the same time frame as the other annotations)
onset_offset = raw.first_time if raw.annotations.orig_time is not None else 0.0
window_onsets = onset_offset + np.arange(n_win) * win_s
raw.set_annotations(raw.annotations + mne.Annotations(onset=window_onsets[is_bad], duration=win_s,
                                                      description="BAD_amplitude",
                                                      orig_time=raw.annotations.orig_time))

# 3) how much of each trial is lost
ann = raw.annotations
rows = []
for description, start in zip(ann.description, ann.onset):
    if description.startswith(cfg.TRIAL_START):
        trial = description[len(cfg.TRIAL_START):]                           # e.g. joint/r2/t0
        end = ann.onset[ann.description == cfg.TRIAL_END + trial][0]
        in_trial = (window_onsets >= start) & (window_onsets + win_s <= end)
        rows.append({"trial": trial, "trial_s": round(end - start, 1), "bad_s": float(is_bad[in_trial].sum() * win_s)})
lost = pd.DataFrame(rows)
lost["bad_percent"] = (100 * lost["bad_s"] / lost["trial_s"]).round(1)
lost.insert(0, "subject", SUBJECT)
lost.to_csv(OUT_DIR / f"{SUBJECT}_bad_segments.csv", index=False)

bad_s, trial_s = lost["bad_s"].sum(), lost["trial_s"].sum()
print(f"{SUBJECT}: BAD_amplitude {bad_s:.0f} s of {trial_s:.0f} s of trials ({100 * bad_s / trial_s:.1f}%)")
print(lost[lost["bad_s"] > 0].to_string(index=False) if bad_s > 0 else "No trial lost anything.")

# 4) overview: window SD over time, threshold in red
fig, ax = plt.subplots(figsize=(12, 3))
ax.semilogy(np.arange(n_win) * win_s, window_sd, lw=0.5, color="k")
ax.axhline(cfg.BAD_SEGMENT_SD_UV, color="tab:red", ls="--", lw=0.8)
ax.set(xlabel="time from the start of the cropped data (s)", ylabel="median channel SD (µV)",
       title=f"{SUBJECT}: 1 s windows above the red line are marked BAD_amplitude")
plt.show()

## 4. Bad channels

Bad channels come from three sources: channels marked bad in the dataset, **clearly bad channels found automatically** (`utils.find_bad_channels`: noisy, flat or uncorrelated with every other channel, criteria of the PREP pipeline; thresholds in `config.py`), and any you add by hand to `BAD_CHANNELS` in `config.py` after looking at the data. The printed median channel SD is also a units check: it should be tens of µV (question 18).

Bad channels are **interpolated before ICA**: CORRMAP compares component maps across participants, so every participant needs the same channels.

Then the EEG channels are re-referenced if `cfg.REFERENCE = "average"` (common average). This comes **after** the interpolation, so the rebuilt channels take part in the average. EOG channels are not changed. (Tip: `%matplotlib qt` gives interactive plots.)

In [ ]:
# units check: should be tens of µV
sd = raw.copy().filter(1, 40, picks="eeg", verbose="error").get_data(picks="eeg", reject_by_annotation="omit").std(axis=1)
print(f"Median channel SD (1-40 Hz): {np.median(sd) * 1e6:.1f} µV")

# Browsing the whole recording and its spectrum takes minutes, so it is off by default
# (set SHOW_RAW = True when running this notebook by hand and you want to look)
SHOW_RAW = False
if SHOW_RAW:
    raw.plot(n_channels=30, scalings=dict(eeg=50e-6))
    raw.compute_psd(fmax=60).plot()

In [ ]:
import importlib
importlib.reload(cfg)  # pick up changes made to config.py

dataset_bads = list(raw.info["bads"])
channel_check = find_bad_channels(raw, cfg.BAD_CHANNEL_Z, cfg.BAD_CHANNEL_CORR, cfg.BAD_CHANNEL_CORR_FRACTION,
                                  cfg.BAD_CHANNEL_FLAT_UV)
auto_bads = channel_check.loc[channel_check["reason"] != "", "channel"].tolist()
manual_bads = cfg.BAD_CHANNELS.get(SUBJECT, [])
bad_channels = sorted(set(dataset_bads) | set(auto_bads) | set(manual_bads))
raw.info["bads"] = bad_channels
print("Bad channels:", bad_channels, "| dataset:", dataset_bads, "| automatic:", auto_bads, "| by hand:", manual_bads)
print(channel_check.sort_values("amplitude_z", ascending=False).to_string(index=False))
if bad_channels:
    raw.interpolate_bads(reset_bads=True)

# Re-reference (cfg.REFERENCE): "average" = every EEG channel minus the mean of all EEG channels
if cfg.REFERENCE == "average":
    raw.set_eeg_reference("average", projection=False)
print("Reference:", cfg.REFERENCE or "recording reference (mastoids)")

## 5. Saving the prepared data

The cropped data at the original sampling rate (500 Hz), with breaks marked and bad channels interpolated. `03_clean.ipynb` applies the ICA to this file (as `ana01` writes the ICA weights to the original data).

In [ ]:
raw.save(OUT_DIR / f"{SUBJECT}_prep_raw.fif", overwrite=True)

## 6. ICA training copy

Exactly the order of `ana01_ICA.m`: low-pass 40 Hz (order 100) → downsample to 250 Hz → high-pass 1 Hz (order 500). Filters are windowed-sinc FIR with a Hann window, like EEGLAB's `pop_firws`. This copy is only used to learn the ICA.

In [ ]:
raw_ica = raw.copy()
firws_filter(raw_ica, cfg.ICA_LP, "lowpass", cfg.ICA_LP_ORDER)
raw_ica.resample(cfg.ICA_SRATE)
firws_filter(raw_ica, cfg.ICA_HP, "highpass", cfg.ICA_HP_ORDER)

## 7. Dummy epochs and joint-probability rejection

The continuous copy is cut into 1 s pieces ("dummy epochs", `eeg_regepochs`), skipping the breaks and the bad stretches (`BAD_amplitude`, step 3b). Pieces with very unusual values (joint probability more than 3 SD from the mean, per channel or over all channels, as `pop_jointprob`) are removed, so large artifacts do not dominate the ICA.

In [ ]:
epochs = mne.make_fixed_length_epochs(raw_ica, duration=cfg.ICA_EPOCH_LENGTH, reject_by_annotation=True,
                                      preload=True)
keep = jointprob_keep(epochs.get_data(picks="eeg"), cfg.ICA_PRUNE)
epochs_ica = epochs[np.where(keep)[0]]
print(f"Dummy epochs: {len(epochs)}, rejected {np.sum(~keep)} ({100 * np.mean(~keep):.1f}%), kept {len(epochs_ica)}")

## 8. ICA

Extended Infomax (EEGLAB's `runica` with `'extended', 1`), on the EEG channels. The PCA option of the reference keeps 50 dimensions; it is capped at the **rank** of the data (the number of independent signals in it), because there are fewer than 50 channels:

rank = EEG channels − interpolated channels − 1 if common average reference

- each interpolated channel is a mix of its neighbours, so it adds no new signal;
- after a common average reference the channels sum to zero, so one channel can be computed from the others.

The rank estimated from the data (`mne.compute_rank`) is printed as a check: it should be the same number. The number of iterations is saved: if it equals the limit, the ICA stopped before it converged.

In [ ]:
n_eeg = len(mne.pick_types(raw_ica.info, eeg=True))
rank = n_eeg - len(bad_channels) - (1 if cfg.REFERENCE == "average" else 0)
n_components = min(cfg.ICA_PCADIMS, rank) if cfg.ICA_PCA else rank
rank_from_data = mne.compute_rank(epochs_ica, rank=None, verbose="error")["eeg"]
print(f"EEG channels: {n_eeg}, interpolated: {len(bad_channels)}, reference: {cfg.REFERENCE or 'mastoids'}")
print(f"Rank: {rank} (estimated from the data: {rank_from_data}), ICA components: {n_components}")

ica = ICA(
    n_components=n_components,
    method="infomax",
    fit_params=dict(extended=True),  # extended Infomax = EEGLAB runica 'extended', 1
    max_iter="auto",
    random_state=cfg.RANDOM_STATE,
)
ica.fit(epochs_ica, picks="eeg")
print(ica, f"| iterations: {ica.n_iter_}")

## 9. Saving the ICA and the component plots

The ICA solution is saved for `02_corrmap.ipynb`. The component topographies are saved as PNG (as in `ana02`, part 1): use them to choose the CORRMAP templates.

In [ ]:
ica.save(OUT_DIR / f"{SUBJECT}_ica.fif", overwrite=True)

# all maps in one PNG; 02_corrmap redraws it with the removed components in red
for old in OUT_DIR.glob(f"{SUBJECT}_ica_components_*.png"):  # several pages, from an earlier version
    old.unlink()
figure = plot_component_maps(ica, SUBJECT)
figure.savefig(OUT_DIR / f"{SUBJECT}_ica_components.png", dpi=100)
plt.close(figure)

pd.DataFrame([{
    "subject": SUBJECT,
    "bad_channels": " ".join(bad_channels),
    "n_bad_channels": len(bad_channels),
    "bad_channels_dataset": " ".join(dataset_bads),
    "bad_channels_auto": " ".join(auto_bads),
    "break_s": round(float(np.sum(break_durations)), 1),
    "bad_amplitude_s": bad_s,
    "bad_amplitude_percent": round(100 * bad_s / trial_s, 1),
    "n_acq_gaps": n_acq_gaps,
    "n_dummy_epochs": len(epochs),
    "n_rejected_epochs": int(np.sum(~keep)),
    "reference": cfg.REFERENCE or "mastoids",
    "rank_from_data": rank_from_data,
    "n_ica_components": ica.n_components_,
    "n_ica_iterations": ica.n_iter_,
}]).to_csv(OUT_DIR / f"{SUBJECT}_ica_info.csv", index=False)
print("Saved to", OUT_DIR)

## 10. Property plot of every component

Map, time course (1 s pieces) and spectrum of each component, on the ICA training copy (1–40 Hz, breaks left out), as PNG in `components/`. These are the same kind of plots as in `04`, but for **all** components and straight after the ICA, so a change in the pipeline (reference, bad channels, ...) can be judged without running CORRMAP and cleaning first.

In [ ]:
COMPONENT_DIR = OUT_DIR / "components"
COMPONENT_DIR.mkdir(exist_ok=True)
for old in COMPONENT_DIR.glob("*.png"):  # plots of an earlier run
    old.unlink()

for comp in range(ica.n_components_):
    figure = ica.plot_properties(raw_ica, picks=[comp], psd_args=dict(fmax=40), show=False)[0]
    figure.suptitle(f"{SUBJECT} ICA{comp:03d}", fontsize=9, y=1.04)
    figure.savefig(COMPONENT_DIR / f"{SUBJECT}_ICA{comp:03d}.png", dpi=100, bbox_inches="tight")
    plt.close(figure)
print(f"{ica.n_components_} property plots saved in {COMPONENT_DIR}")